In [1]:
# 1. Загрузка сырого слоя
import pandas as pd
from pathlib import Path

RAW = Path("ege_all.csv")        # если ноутбук в notebooks/ → Path("..") / "ege_all.csv"
df = pd.read_csv(RAW, sep=";", encoding="utf-8-sig", low_memory=False)
assert df.shape == (167974, 25), df.shape
df.head(3)

,category,year,admission_type,university,field_group,region,profile,avg_score,score_change,min_score,...,enrolled_competition,enrolled_bvi,enrolled_quota,enrolled_target,enrolled_crimea,enrolled_budget,n_known_scores,tuition,id_deducted,source_file
0,Данные по направлениям подготовки,2011,Бюджетный прием,Моск. физико-техн. ин-т. (МФТИ),Математика,Москва и Московская область,NaN,89.4,NaN,79.7,...,75.0,13.0,2.0,NaN,NaN,NaN,75.0,NaN,NaN,napravleniya_2011_budget.csv
1,Данные по направлениям подготовки,2011,Бюджетный прием,Моск. гос. ун-т. им. М.В. Ломоносова,Математика,Москва и Московская область,NaN,87.6,NaN,70.7,...,577.0,148.0,29.0,NaN,NaN,NaN,577.0,NaN,NaN,napravleniya_2011_budget.csv
2,Данные по направлениям подготовки,2011,Бюджетный прием,"Национальный исследовательский ун-т. ""Высшая ш...",Математика,Москва и Московская область,NaN,85.2,NaN,80.0,...,78.0,52.0,2.0,NaN,NaN,NaN,78.0,NaN,NaN,napravleniya_2011_budget.csv


In [2]:
# 2. Осмотр
print(df.dtypes)
print("\nПропуски по колонкам:\n", df.isna().sum())
print("\nСтроки по группам:\n", df.groupby(["category", "admission_type"]).size())

category                        str
year                          int64
admission_type                  str
university                      str
field_group                     str
region                          str
profile                         str
avg_score                   float64
score_change                float64
min_score                   float64
avg_score_competition       float64
score_change_competition    float64
avg_score_no_id             float64
avg_score_budget            float64
enrolled                    float64
enrolled_competition        float64
enrolled_bvi                float64
enrolled_quota              float64
enrolled_target             float64
enrolled_crimea             float64
enrolled_budget             float64
n_known_scores              float64
tuition                     float64
id_deducted                  object
source_file                     str
dtype: object

Пропуски по колонкам:
 category                         0
year                       

In [4]:
# 3. Документированная очистка (все шаги проверены, см. DATA_NOTES.md)
BROKEN = ["vuzy_2012_paid.csv", "vuzy_2013_paid.csv"]

df["id_deducted"] = (df["id_deducted"].astype(str).str.strip()
                     .map({"True": True, "False": False}).astype("boolean"))
df.loc[df["tuition"] < 2000, "tuition"] = pd.NA          # п.2
df["qc_unreliable"] = df["source_file"].isin(BROKEN)      # п.3
df.loc[df["qc_unreliable"], ["min_score", "enrolled"]] = pd.NA
df["coverage"] = df["year"].map(lambda y: "state_only" if y <= 2017 else "all")  # п.1

assert len(df) == 167974
assert df["qc_unreliable"].sum() == 461
assert df["tuition"].min() >= 2000

In [5]:
# 4. Диагностика названий — с этого начнём нормализацию
v = df[df.category == "Данные по вузам"]
n_years = v.groupby("university")["year"].nunique()
print("уникальных названий:", len(n_years))
print("вузов во все 15 лет:", (n_years == 15).sum())
print(n_years.sort_values(ascending=False).head(10))

уникальных названий: 1385
вузов во все 15 лет: 155
university
Адыгейский гос. ун-т., г. Майкоп                                                        15
Дагестанский гос. педагогический ун-т., г. Махачкала                                    15
Дальневосточный гос. аграрный ун-т., г. Благовещенск                                    15
Северо-Кавказский горно-металлургический ин-т. (гос. технол. ун-т.) , г. Владикавказ    15
Дальневосточный гос. медицинский ун-т., г. Хабаровск                                    15
Дальневосточный гос. техн. рыбохозяйственный ун-т., г. Владивосток                      15
Дальневосточный гос. ун-т. путей сообщения, г. Хабаровск                                15
Северный гос. медицинский ун-т., г. Архангельск                                         15
Сахалинский гос. ун-т., г. Южно-Сахалинск                                               15
Саратовский гос. медицинский ун-т.                                                      15
Name: year, dtype: int64
